In [ ]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno
from pathlib import Path

from src.db import crear_db_desde_csv, ejecutar_query
from src.eda_utils import (
    resumen_nulos,
    resumen_ceros,
    plot_mapa_nulos,
    plot_heatmap_nulos,
    plot_desbalance_objetivo,
    plot_distribuciones_numericas,
    plot_distribuciones_categoricas,
    plot_correlacion_numerica,
    plot_correlacion_categorica,
    plot_outliers,
    resumen_outliers_iqr,
    ceros_a_nulos,
    eliminar_registros_con_nulos,
    eliminar_registros_con_ceros
)
from src.queries import (
    query_duplicados_exactos,
    query_conteo_nulos,
    query_distribucion_categorica,
    query_desbalance_objetivo,
    query_estadisticas_numericas,
    query_distribucion_por_objetivo
)

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', None)

print("Setup OK")

In [ ]:

RAIZ = Path('..').resolve()
df = pd.read_csv(RAIZ / 'data/raw/hmeq.csv')

crear_db_desde_csv(
    csv_path=RAIZ / 'data/raw/hmeq.csv',
    db_path=RAIZ / 'data/raw/hmeq.db'
)

print(f"Dataset cargado: {df.shape[0]} filas, {df.shape[1]} columnas")
print(f"\nColumnas: {list(df.columns)}")

In [ ]:
# ============================================================
# ANÁLISIS DE CALIDAD — Duplicados exactos
# ============================================================
duplicados = ejecutar_query(query_duplicados_exactos(), db_path=RAIZ / 'data/raw/hmeq.db')
print(f"Filas duplicadas encontradas: {len(duplicados)}")
duplicados

In [ ]:
# ============================================================
# ANÁLISIS DE DATOS FALTANTES
# ============================================================
resumen_nulos(df)

In [ ]:
ejecutar_query(query_conteo_nulos(), db_path=RAIZ / 'data/raw/hmeq.db')

In [ ]:
resumen_ceros(df)

De esta primera exploración podemos ver que: 
- No hay duplicados
- Los resultados de SQL y Pandas coinciden, lo que confirma que la carga a SQLite no se corrompió.
- DEBTINC es candidata a variable de alto poder predictivo según análisis previo — se confirmará en el análisis de correlación, tiene un 21.26% de nulos, casi 1 de cada 5 registros.
- La variable BAD tiene 80.05% de ceros, pero es correcto porque tiene dos valores posibles 0 = pago y 1 = incumplió. Pero al mismo tiempo ya anticipa el desbalance de las clases. 
- Variables con altos porcentajes de ceros : DEROG 75.96%, DELINQ 70.12% y NINQ 42.47%. 
Teniendo en cuenta que DEROG representa los incumplimientos graves, puede ser un valor correcto 0 (cero) con personas sin incumplimientos. 
- Lo mismo con DELINQ que es el indicador de morosidad, el analisis y si existe relación entre ambas debería contrastarse. 
- NINQ wue es actividad reciente de busqueda de crédito, tambien es un valor valido. 
- El resto de las variables que presentan valores en cero tambien tienen sentido y no pueden ser considerados error sin continuar explorando los datos. 

In [ ]:
# ============================================================
# PATRÓN DE NULOS — ¿MCAR, MAR o MNAR?
# ============================================================
plot_mapa_nulos(df)

In [ ]:
plot_heatmap_nulos(df)

Podemos ver que hay correlación en la ausencia de datos. Lo que nos posiciona frente a MAR  (Missing At Random condicionado a otras variables)

DEROG + DELINQ + NINQ + CLNO + CLAGE correlacionados en nulos: probablemente son clientes que no tienen historial crediticio previo — nunca pidieron crédito, nunca tuvieron líneas abiertas, nunca aparecieron en registros de morosidad. Para ese tipo de cliente, todos esos campos quedan vacíos en el formulario porque no hay información que completar. No es que el dato se perdió — es que genuinamente no existe todavía.

Como en el analisis anterior vimos que DEROG, DELINQ y NINQ  tienen 70-75% de ceros reales (clientes sin incumplimientos). Si un cliente no tiene historial y el campo está vacío, asumir que sus derogatorios y moras son 0 es una imputación razonable y consistente con la distribución real. De estta manera podemos aprovechar el valor predictivo de estos registros. 

In [ ]:
# ¿Cuántos registros únicos tienen nulo en CLAGE, CLNO, o ambos?
nulos_clage = df['CLAGE'].isnull()
nulos_clno = df['CLNO'].isnull()
ceros_clage = df['CLAGE'] == 0
ceros_clno = df['CLNO'] == 0

afectados = df[nulos_clage | ceros_clage | nulos_clno | ceros_clno]
print(f"Registros afectados por nulos/ceros en CLAGE o CLNO: {len(afectados)}")
print(f"Porcentaje del dataset: {len(afectados)/len(df)*100:.1f}%")

# De esos registros, ¿cuántos también tienen nulos en DEROG/DELINQ/NINQ?
print(f"\nDe esos registros, nulos en:")
print(f"  DEROG:  {afectados['DEROG'].isnull().sum()}")
print(f"  DELINQ: {afectados['DELINQ'].isnull().sum()}")
print(f"  NINQ:   {afectados['NINQ'].isnull().sum()}")
print(f"  DEBTINC: {afectados['DEBTINC'].isnull().sum()}")

In [ ]:
# ¿Los clientes sin historial incumplen más o menos que el promedio?
sin_historial = df[nulos_clage | ceros_clage | nulos_clno | ceros_clno]
con_historial = df[~(nulos_clage | ceros_clage | nulos_clno | ceros_clno)]

print(f"Tasa de incumplimiento (BAD=1):")
print(f"  Clientes SIN historial (n={len(sin_historial)}): {sin_historial['BAD'].mean()*100:.1f}%")
print(f"  Clientes CON historial (n={len(con_historial)}): {con_historial['BAD'].mean()*100:.1f}%")
print(f"  Dataset completo:                               {df['BAD'].mean()*100:.1f}%")

**Patrón de nulos identificado:**
Existe un grupo de 312 registros (5.2% del dataset) con nulos simultáneos
en CLAGE, CLNO, DEROG, DELINQ y NINQ — consistente con clientes sin historial
crediticio previo (thin file). Este grupo tiene una tasa de incumplimiento
del 26.3% vs. 19.6% del resto, lo que indica que la ausencia de historial
es en sí misma una señal de riesgo. **No se eliminan estos registros.**

**DEBTINC (21.26% de nulos):** patrón de ausencia independiente del resto,
consistente con MNAR (el dato puede faltar en función de su propio valor).
Estrategia: imputar con mediana + crear variable indicadora `DEBTINC_era_nulo`

**Estrategia de imputación definida (a implementar en data_prep.py):**
- DEROG, DELINQ, NINQ → imputar con 0 (ausencia = sin incumplimientos)
- CLNO → imputar con 0 (sin líneas de crédito)
- CLAGE → imputar con mediana de clientes con historial
- REASON, JOB → imputar con moda o categoría 'Desconocido'
- DEBTINC → imputar con mediana + crear `DEBTINC_era_nulo`
- Crear variable `sin_historial` para capturar el perfil thin file
- MORTDUE, VALUE, YOJ → evaluar en análisis de distribución